# پروژه عملی هفته دوم: نسخه «CNN سبک» (Colored MNIST)

این نوت‌بوک همان پروژه قبلی است با یک تغییر معماری: **لایه conv3 حذف شده** و مدل فقط دو لایه کانولوشن دارد (ورودی fc1 برابر 64×7×7 = 3136). هدف، ساخت مدلی است که احتمالاً وابستگی بیشتری به رنگ نشان دهد.

**نکات مهم:**
1. تعداد پارامترهای این مدل **825,034** است (نه ۸۶۳٬۱۷۸ که در طرح آمده بود؛ سلول مدل مقدار واقعی را چاپ می‌کند).
2. اعداد «انتظار از نتایج» (ID ≈ 99.15، OOD-1 ≈ 88، OOD-2 ≈ 45) فرضیه‌اند، نه نتیجه. هر عددی که اجرا بدهد همان گزارش می‌شود.
3. اگر معماری را بعد از دیدن نتایج نسخه اول تغییر داده‌اید، در گزارش **هر دو نسخه** را بیاورید و دلیل تغییر را صریح بنویسید (نسخه اول: ۳ لایه کانولوشن، نسخه دوم: ۲ لایه). گزارش‌کردن فقط نسخه‌ای که به نتیجه دلخواه رسید، انتخاب گزینشی است.
4. Reweighting این‌جا با نمونه‌گیری وزن‌دار (معادل WeightedRandomSampler) انجام می‌شود.
5. خروجی‌ها در پوشه `results_light/` و فایل `project_light_results.zip` ذخیره می‌شوند و نتایج نسخه اول را بازنویسی نمی‌کنند.

**اجرا:** Colab، Runtime → GPU، سپس Run all (حدود ۳۰ تا ۴۰ دقیقه؛ با `QUICK_TEST = True` چند دقیقه).

In [ ]:
import os, json, time, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
import matplotlib.pyplot as plt
import pandas as pd

# ---------------- تنظیمات کلی ----------------
QUICK_TEST = False          # True → تست سریع (۲ epoch، یک seed) فقط برای اطمینان از اجرای بدون خطا
SEEDS      = [0] if QUICK_TEST else [0, 1, 2]
EPOCHS     = 2 if QUICK_TEST else 20
BATCH      = 64
LR         = 1e-3

# هایپرپارامترهای روش‌های کاهش سوگیری (از پیش تعیین شده؛ روی داده OOD تنظیم نشده‌اند)
AUG_FRAC    = 0.10          # روش ۱: نسبت نمونه‌های بدون سوگیری اضافه‌شده (ρ=0.5)
DRO_ETA     = 0.01          # روش ۳: نرخ به‌روزرسانی وزن گروه‌ها (η_q)
GCE_Q       = 0.7           # روش ۴: پارامتر q در Generalized Cross Entropy
EMA_ALPHA   = 0.7           # روش ۴: ضریب EMA برای خطای هر نمونه
IRM_LAMBDA  = 1e4           # روش ۵: وزن جریمه IRM بعد از دوره گرم‌کردن
IRM_ANNEAL  = 1 if QUICK_TEST else 5   # روش ۵: تعداد epochهای اولیه با λ=1

OUT = "results_light"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARCH_FA = "CNN سبک با دو لایه کانولوشن ۳×۳ (۳۲ و ۶۴ فیلتر) با ReLU و یک MaxPool بعد از لایه اول، یک لایه کاملاً متصل ۲۵۶ نورونی (ورودی ۳۱۳۶)، Dropout با نرخ ۰.۵ و لایه خروجی ۱۰ کلاسه (بدون conv3)"
print("device:", DEVICE, "| torch", torch.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

## ۱. ساخت Colored MNIST

In [ ]:
# ---------------- ساخت Colored MNIST ----------------
mn_tr = datasets.MNIST('./data', train=True,  download=True)
mn_te = datasets.MNIST('./data', train=False, download=True)
G_tr = mn_tr.data.numpy().astype(np.float32) / 255.0      # (60000, 28, 28)
Y_tr = mn_tr.targets.numpy().astype(np.int64)
G_te = mn_te.data.numpy().astype(np.float32) / 255.0      # (10000, 28, 28)
Y_te = mn_te.targets.numpy().astype(np.int64)

def make_colors(y, rho, seed):
    """رنگ هر نمونه: 0=قرمز، 1=سبز.
    ارقام 0-4 با احتمال rho قرمز (و در غیر این صورت سبز)، ارقام 5-9 با احتمال rho سبز."""
    rng  = np.random.RandomState(seed)
    base = (y >= 5).astype(np.int64)                 # رنگ «هم‌جهت» با شکل
    keep = rng.rand(len(y)) < rho
    return np.where(keep, base, 1 - base)

class CData:
    """مجموعه داده روی GPU: تصویر RGB، برچسب، رنگ، نیمه‌ی کلاس و گروه (رنگ×نیمه‌ی کلاس)."""
    def __init__(self, gray, y, color):
        n   = len(y)
        rgb = np.zeros((n, 3, 28, 28), np.float32)
        rgb[np.arange(n), color] = gray              # کانال 0=R، 1=G
        self.x = torch.from_numpy(rgb).to(DEVICE)
        self.y = torch.from_numpy(y).long().to(DEVICE)
        self.c = torch.from_numpy(color).long().to(DEVICE)
        self.half    = (self.y >= 5).long()          # 0: ارقام 0-4 ، 1: ارقام 5-9
        self.g       = self.c * 2 + self.half        # 0:قرمز/0-4  1:قرمز/5-9  2:سبز/0-4  3:سبز/5-9
        self.aligned = (self.c == self.half)         # رنگ هم‌جهت با شکل؟
    def __len__(self):
        return len(self.y)

def make_cmnist(gray, y, rho, seed):
    return CData(gray, y, make_colors(y, rho, seed))

# تقسیم ارقام: آموزش/اعتبارسنجی از train، و همه ۱۰٬۰۰۰ رقم test برای سه مجموعه تست
perm   = np.random.RandomState(42).permutation(len(G_tr))
tr_idx = perm[:50000]
va_idx = perm[50000:]

S = {
    'train': make_cmnist(G_tr[tr_idx], Y_tr[tr_idx], 0.9, 42),
    'val':   make_cmnist(G_tr[va_idx], Y_tr[va_idx], 0.9, 43),
    'id':    make_cmnist(G_te, Y_te, 0.9, 44),
    'ood1':  make_cmnist(G_te, Y_te, 0.5, 45),
    'ood2':  make_cmnist(G_te, Y_te, 0.1, 46),
}
assert len(set(tr_idx) & set(va_idx)) == 0            # بدون نشت بین آموزش و اعتبارسنجی

SET_INFO = {   # برای جدول ۱
    'train': ('آموزش', 0.9), 'val': ('اعتبارسنجی', 0.9), 'id': ('تست ID', 0.9),
    'ood1': ('تست OOD-1 (بدون سوگیری)', 0.5), 'ood2': ('تست OOD-2 (معکوس)', 0.1)}
print(f"{'set':6s} {'N':>6s}  fraction(color aligned with shape)")
for k, d in S.items():
    print(f"{k:6s} {len(d):6d}  {d.aligned.float().mean().item():.3f}")

# ---------------- نمونه تصاویر ----------------
fig, axes = plt.subplots(5, 10, figsize=(13, 7))
for r, (k, d) in enumerate(S.items()):
    for j in range(10):
        i = torch.nonzero(d.y == j)[0, 0].item()
        axes[r, j].imshow(d.x[i].permute(1, 2, 0).cpu().numpy()); axes[r, j].axis('off')
    axes[r, 0].set_title(f"{k} (rho={SET_INFO[k][1]})", loc='left', fontsize=9)
plt.suptitle("Colored MNIST samples (one per digit)"); plt.tight_layout()
plt.savefig(f"{OUT}/fig0_dataset_samples.png", dpi=150); plt.show()

## ۲. مدل (CNN سبک) و ابزارها

In [ ]:
# ---------------- مدل و ابزارهای ارزیابی ----------------
class SimpleCNN(nn.Module):
    """CNN سبک: فقط دو لایه کانولوشن (conv3 حذف شده). features/head برای Grad-CAM جدا شده‌اند."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 256)      # 64*7*7 = 3136
        self.fc2   = nn.Linear(256, 10)
        self.dropout = nn.Dropout(0.5)
    def features(self, x):
        x = self.pool(F.relu(self.conv1(x)))         # 14x14
        return F.relu(self.conv2(x))                 # 64x7x7 (آخرین لایه کانولوشن)
    def head(self, f):
        x = f.reshape(f.size(0), -1)                 # (B, 3136)  ← f، نه x
        x = self.dropout(F.relu(self.fc1(x)))        # (B, 256)
        return self.fc2(x)                           # (B, 10)
    def forward(self, x):
        return self.head(self.features(x))

# تست سریع مدل قبل از آموزش
_t = SimpleCNN().to(DEVICE)
assert _t(torch.randn(4, 3, 28, 28, device=DEVICE)).shape == (4, 10)
print(f"Total parameters: {sum(p.numel() for p in _t.parameters()):,}")   # باید 825,034 باشد
del _t

@torch.no_grad()
def predict(model, x, bs=5000):
    model.eval()
    return torch.cat([model(x[i:i+bs]).argmax(1) for i in range(0, len(x), bs)])

@torch.no_grad()
def eval_loss_acc(model, d, bs=5000):
    model.eval(); L, C = 0.0, 0
    for i in range(0, len(d), bs):
        lo, y = model(d.x[i:i+bs]), d.y[i:i+bs]
        L += F.cross_entropy(lo, y, reduction='sum').item()
        C += (lo.argmax(1) == y).sum().item()
    return L / len(d), C / len(d)

def new_hist():
    return {k: [] for k in ['obj', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'ood1_acc', 'ood2_acc']}

def log_epoch(h, model, obj, tag, ep):
    """ثبت منحنی‌ها روی مجموعه آموزش استاندارد (ρ=0.9) و اعتبارسنجی.
    دقت OOD فقط برای رسم پویایی یادگیری ثبت می‌شود و هرگز برای انتخاب مدل/هایپرپارامتر استفاده نمی‌شود."""
    tl, ta = eval_loss_acc(model, S['train']); vl, va = eval_loss_acc(model, S['val'])
    h['obj'].append(float(obj)); h['train_loss'].append(tl); h['train_acc'].append(ta)
    h['val_loss'].append(vl);    h['val_acc'].append(va)
    h['ood1_acc'].append(eval_loss_acc(model, S['ood1'])[1])
    h['ood2_acc'].append(eval_loss_acc(model, S['ood2'])[1])
    if (ep + 1) % 5 == 0 or ep == 0:
        print(f"  [{tag}] epoch {ep+1:2d}  obj={obj:.4f}  train_acc={ta:.4f}  val_acc={va:.4f}  "
              f"ood1={h['ood1_acc'][-1]:.4f}  ood2={h['ood2_acc'][-1]:.4f}")

def final_metrics(model):
    m = {k: eval_loss_acc(model, S[k])[1] for k in ['train', 'val', 'id', 'ood1', 'ood2']}
    d = S['id']; ok = (predict(model, d.x) == d.y)
    m['id_aligned']  = ok[d.aligned].float().mean().item()     # رنگ هم‌جهت با شکل
    m['id_conflict'] = ok[~d.aligned].float().mean().item()    # رنگ مخالف شکل
    return m

RES, HIST, MODELS, CUE = {}, {}, {}, {}     # نتایج: RES[method][seed] ...

def register(method, seed, model, hist):
    RES.setdefault(method, {})[seed]   = final_metrics(model)
    HIST.setdefault(method, {})[seed]  = hist
    MODELS.setdefault(method, {})[seed] = model
    r = RES[method][seed]
    print(f"==> {method} seed={seed}: ID={r['id']:.4f} OOD1={r['ood1']:.4f} OOD2={r['ood2']:.4f}")

def save_results():
    cfg = dict(EPOCHS=EPOCHS, BATCH=BATCH, LR=LR, SEEDS=SEEDS, AUG_FRAC=AUG_FRAC, DRO_ETA=DRO_ETA,
               GCE_Q=GCE_Q, EMA_ALPHA=EMA_ALPHA, IRM_LAMBDA=IRM_LAMBDA, IRM_ANNEAL=IRM_ANNEAL, ARCH_FA=ARCH_FA,
               N_TRAIN=len(S['train']), DEVICE=DEVICE, QUICK_TEST=QUICK_TEST)
    with open(f"{OUT}/results.json", "w", encoding="utf-8") as f:
        json.dump(dict(cfg=cfg, RES=RES, CUE=CUE, HIST=HIST), f, ensure_ascii=False)

## وظیفه ۱: آموزش مدل پایه

In [ ]:
# ---------------- وظیفه ۱: آموزش مدل پایه ----------------
def fit_erm(tr, seed, weights=None, sample_w=None, epochs=EPOCHS, tag='ERM'):
    """آموزش استاندارد (ERM). weights: وزن‌دهی خطا؛ sample_w: نمونه‌گیری وزن‌دار (معادل WeightedRandomSampler) در هر epoch."""
    set_seed(seed)
    model = SimpleCNN().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    h, N  = new_hist(), len(tr)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(N, device=DEVICE) if sample_w is None else torch.multinomial(sample_w, N, replacement=True)
        run, nb = torch.zeros((), device=DEVICE), 0
        for i in range(0, N, BATCH):
            idx  = perm[i:i+BATCH]
            ce   = F.cross_entropy(model(tr.x[idx]), tr.y[idx], reduction='none')
            loss = ce.mean() if weights is None else (weights[idx] * ce).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            run += loss.detach(); nb += 1
        log_epoch(h, model, (run / nb).item(), tag, ep)
    return model, h

t0 = time.time()
for s in SEEDS:
    m, h = fit_erm(S['train'], s, tag='Baseline')
    register('Baseline', s, m, h)
print(f"time: {time.time()-t0:.0f}s")

# ذخیره وزن‌ها (seed اول)
torch.save(MODELS['Baseline'][SEEDS[0]].state_dict(), "baseline_model.pth")
torch.save(MODELS['Baseline'][SEEDS[0]].state_dict(), f"{OUT}/baseline_model.pth")

# منحنی Loss و Accuracy (seed اول)
h = HIST['Baseline'][SEEDS[0]]; ep = np.arange(1, len(h['train_loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, h['train_loss'], label='train'); ax[0].plot(ep, h['val_loss'], label='validation')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('cross-entropy loss'); ax[0].set_title('Baseline: loss'); ax[0].legend()
ax[1].plot(ep, h['train_acc'], label='train'); ax[1].plot(ep, h['val_acc'], label='validation')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].set_title('Baseline: accuracy'); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/fig1_baseline_training_curves.png", dpi=150); plt.show()
print("ID accuracy (mean over seeds): %.2f%%" % (100*np.mean([RES['Baseline'][s]['id'] for s in SEEDS])))
save_results()

## وظیفه ۲: ارزیابی ID/OOD (جدول ۱)

In [ ]:
# ---------------- وظیفه ۲: ارزیابی ID و OOD (جدول ۱) ----------------
def agg(method, key, src=None):
    src = RES if src is None else src
    v = np.array([src[method][s][key] for s in SEEDS]) * 100
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)

rows = []
for k in ['train', 'val', 'id', 'ood1', 'ood2']:
    m, sd = agg('Baseline', k)
    rows.append({'مجموعه': SET_INFO[k][0], 'ρ': SET_INFO[k][1], 'دقت % (mean±std)': f"{m:.2f} ± {sd:.2f}"})
for k, nm in [('id_aligned', 'تست ID – رنگ هم‌جهت با شکل'), ('id_conflict', 'تست ID – رنگ مخالف شکل')]:
    m, sd = agg('Baseline', k)
    rows.append({'مجموعه': nm, 'ρ': '-', 'دقت % (mean±std)': f"{m:.2f} ± {sd:.2f}"})
T1 = pd.DataFrame(rows); T1.to_csv(f"{OUT}/table1_baseline_eval.csv", index=False, encoding='utf-8-sig')
display(T1)

fig, ax = plt.subplots(figsize=(7, 4))
keys = ['train', 'val', 'id', 'ood1', 'ood2']
mm = [agg('Baseline', k)[0] for k in keys]; ss = [agg('Baseline', k)[1] for k in keys]
ax.bar(['train', 'val', 'ID', 'OOD-1', 'OOD-2'], mm, yerr=ss, capsize=4, color=['C0', 'C0', 'C2', 'C1', 'C3'])
for i, v in enumerate(mm): ax.text(i, v + 1, f"{v:.1f}", ha='center')
ax.set_ylabel('accuracy (%)'); ax.set_ylim(0, 110); ax.set_title('Baseline accuracy on the five sets')
plt.tight_layout(); plt.savefig(f"{OUT}/fig2_baseline_accuracy.png", dpi=150); plt.show()

## وظیفه ۳: تشخیص نشانه (جدول ۲ و ۲ب)

In [ ]:
# ---------------- وظیفه ۳: تشخیص نشانه ----------------
# ۳.۱ آزمون‌های Counterfactual:  تست A (تعویض رنگ)  و  تست B (حذف رنگ)
def swap_rg(x):    return x[:, [1, 0, 2]]                                    # قرمز↔سبز
def gray_mean(x):  return x.mean(1, keepdim=True).expand(-1, 3, -1, -1).clone()   # طبق صورت پروژه (شدت ۱/۳ می‌شود)
def gray_sum(x):   return x.sum(1, keepdim=True).expand(-1, 3, -1, -1).clone()    # کنترل: حذف رنگ با حفظ شدت

# ۳.۴ تغییر نشانه شکل (Misspecification)
def rot90(x):         return torch.rot90(x, 1, dims=(2, 3))
def occlude_top(x):
    x = x.clone(); x[:, :, :14, :] = 0; return x                              # حذف نیمه بالایی رقم
def patch_shuffle(x, p=7):
    B, C, H, W = x.shape; g = H // p
    pt  = x.unfold(2, p, p).unfold(3, p, p)                                   # (B,C,g,g,p,p)
    pt  = pt.reshape(B, C, g * g, p, p).permute(0, 2, 1, 3, 4)                 # (B,n,C,p,p)
    prm = torch.argsort(torch.rand(B, g * g, device=x.device), dim=1)
    pt  = pt[torch.arange(B, device=x.device)[:, None], prm]                  # جابه‌جایی تصادفی قطعه‌ها
    return pt.reshape(B, g, g, C, p, p).permute(0, 3, 1, 4, 2, 5).reshape(B, C, H, W)

def probe(model, x, y, c_shown):
    """acc: دقت ۱۰ کلاسه | half_acc: دقت تشخیص نیمه‌ی کلاس (0-4 یا 5-9)
    follow: نرخ پیروی از رنگ = درصد پیش‌بینی‌هایی که نیمه‌ی کلاسشان با نیمه‌ی القاشده توسط رنگ نمایش‌داده‌شده یکی است."""
    p = predict(model, x)
    return {'acc':      (p == y).float().mean().item(),
            'half_acc': ((p >= 5).long() == (y >= 5).long()).float().mean().item(),
            'follow':   ((p >= 5).long() == c_shown).float().mean().item()}

g_sub = torch.Generator().manual_seed(7)
SUB   = torch.randperm(len(S['id']), generator=g_sub)[:1000].to(DEVICE)      # ۱۰۰۰ نمونه تست برای تست Misspecification

def cue_suite(model):
    d, o = S['id'], {}
    o['original']        = probe(model, d.x, d.y, d.c)
    o['A_swap']          = probe(model, swap_rg(d.x),   d.y, 1 - d.c)
    o['B_gray_mean']     = probe(model, gray_mean(d.x), d.y, d.c)
    o['B_gray_sum']      = probe(model, gray_sum(d.x),  d.y, d.c)
    xs, ys, cs = d.x[SUB], d.y[SUB], d.c[SUB]
    torch.manual_seed(1234)
    o['M_orig']          = probe(model, xs, ys, cs)
    o['M_shape_rot90']   = probe(model, rot90(xs), ys, cs)
    o['M_shape_occlude'] = probe(model, occlude_top(xs), ys, cs)
    o['M_shape_shuffle'] = probe(model, patch_shuffle(xs), ys, cs)
    o['M_color_swap']    = probe(model, swap_rg(xs), ys, 1 - cs)
    o['M_color_gray']    = probe(model, gray_sum(xs), ys, cs)
    return o

for s in SEEDS:
    CUE.setdefault('Baseline', {})[s] = cue_suite(MODELS['Baseline'][s])

def cagg(method, probe_name, key):
    v = np.array([CUE[method][s][probe_name][key] for s in SEEDS]) * 100
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)

LABELS = [('original', 'تست ID اصلی (مرجع)'), ('A_swap', 'تست A: تعویض رنگ'),
          ('B_gray_mean', 'تست B: Grayscale (میانگین کانال‌ها، طبق صورت پروژه)'),
          ('B_gray_sum', 'تست B′: Grayscale با حفظ شدت (کنترل)')]
rows = []
for k, nm in LABELS:
    m, sd = cagg('Baseline', k, 'acc'); f, _ = cagg('Baseline', k, 'follow')
    rows.append({'تست': nm, 'دقت %': f"{m:.2f} ± {sd:.2f}", 'نرخ پیروی از رنگ %': f"{f:.1f}"})
T2 = pd.DataFrame(rows); T2.to_csv(f"{OUT}/table2_counterfactual.csv", index=False, encoding='utf-8-sig'); display(T2)

LABELS_M = [('M_orig', 'بدون تغییر (۱۰۰۰ نمونه)'), ('M_shape_rot90', 'تغییر شکل: چرخش ۹۰°'),
            ('M_shape_occlude', 'تغییر شکل: حذف نیمه بالایی'), ('M_shape_shuffle', 'تغییر شکل: بُر زدن قطعه‌ها'),
            ('M_color_swap', 'تغییر رنگ: تعویض'), ('M_color_gray', 'تغییر رنگ: حذف (حفظ شدت)')]
rows = []
for k, nm in LABELS_M:
    a, sd = cagg('Baseline', k, 'acc'); h_, _ = cagg('Baseline', k, 'half_acc'); f, _ = cagg('Baseline', k, 'follow')
    rows.append({'آزمون': nm, 'دقت ۱۰ کلاسه %': f"{a:.2f} ± {sd:.2f}", 'دقت نیمه‌کلاس %': f"{h_:.1f}", 'نرخ پیروی از رنگ %': f"{f:.1f}"})
T2b = pd.DataFrame(rows); T2b.to_csv(f"{OUT}/table2b_misspecification.csv", index=False, encoding='utf-8-sig'); display(T2b)
save_results()

### Saliency Map

In [ ]:
# ۳.۲ Saliency Map  (Gradient × Input) + گرادیان خام به تفکیک کانال رنگ
def attributions(model, x, target):
    model.eval()
    x = x.clone().requires_grad_(True)
    score = model(x).gather(1, target[:, None]).sum()
    grad, = torch.autograd.grad(score, x)
    return grad.detach(), (grad * x).detach()

base_model = MODELS['Baseline'][SEEDS[0]]
d = S['id']
idx10 = [torch.nonzero(d.y == k)[0, 0].item() for k in range(10)]            # یک نمونه از هر کلاس
x, y  = d.x[idx10], d.y[idx10]
g, gxi = attributions(base_model, x, y)
pred = predict(base_model, x)

fig, axes = plt.subplots(4, 10, figsize=(18, 7.6))
for j in range(10):
    axes[0, j].imshow(x[j].permute(1, 2, 0).cpu().numpy())
    axes[0, j].set_title(f"y={y[j].item()}  pred={pred[j].item()}", fontsize=9)
    axes[1, j].imshow(gxi[j].abs().sum(0).cpu().numpy(), cmap='hot')
    v = g[j].abs().max().item() + 1e-12
    axes[2, j].imshow(g[j, 0].cpu().numpy(), cmap='RdBu_r', vmin=-v, vmax=v)
    axes[3, j].imshow(g[j, 1].cpu().numpy(), cmap='RdBu_r', vmin=-v, vmax=v)
    for r in range(4): axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
for r, t in enumerate(['input', '|Gradient x Input|', 'gradient: R channel', 'gradient: G channel']):
    axes[r, 0].set_ylabel(t, fontsize=9)
plt.suptitle("Saliency maps of the baseline model (true-class logit)"); plt.tight_layout()
plt.savefig(f"{OUT}/fig3_saliency.png", dpi=150); plt.show()

# شکل کامل: ۱۰ نمونه از هر کلاس (تصویر + Saliency)
fig, axes = plt.subplots(10, 20, figsize=(20, 10))
for k in range(10):
    ids = torch.nonzero(d.y == k)[:10, 0]
    gk, gxik = attributions(base_model, d.x[ids], d.y[ids])
    for j in range(10):
        axes[k, 2*j].imshow(d.x[ids[j]].permute(1, 2, 0).cpu().numpy())
        axes[k, 2*j+1].imshow(gxik[j].abs().sum(0).cpu().numpy(), cmap='hot')
        axes[k, 2*j].axis('off'); axes[k, 2*j+1].axis('off')
plt.suptitle("Gradient x Input for 10 samples per class (image | saliency)"); plt.tight_layout()
plt.savefig(f"{OUT}/fig3b_saliency_100samples.png", dpi=110); plt.show()

### Grad-CAM (روی conv2، نقشه 7×7)

In [ ]:
# ۳.۳ Grad-CAM روی آخرین لایه کانولوشن (conv3، نقشه 7×7)
def grad_cam(model, x, target=None):
    model.eval()
    f = model.features(x).detach().requires_grad_(True)
    out = model.head(f)
    pred = out.argmax(1)
    t = pred if target is None else target
    grads, = torch.autograd.grad(out.gather(1, t[:, None]).sum(), f)
    w   = grads.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * f).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(28, 28), mode='bilinear', align_corners=False)
    cam = cam / (cam.amax(dim=(2, 3), keepdim=True) + 1e-8)
    return cam.squeeze(1).detach(), pred

x, y = d.x[idx10], d.y[idx10]
cam1, p1 = grad_cam(base_model, x, y)                 # نمونه اصلی، کلاس درست
xs = swap_rg(x)
cam2, p2 = grad_cam(base_model, xs, None)             # همان رقم با رنگ تعویض‌شده، کلاس پیش‌بینی‌شده

fig, axes = plt.subplots(4, 10, figsize=(18, 7.6))
for j in range(10):
    im1, im2 = x[j].permute(1, 2, 0).cpu().numpy(), xs[j].permute(1, 2, 0).cpu().numpy()
    axes[0, j].imshow(im1); axes[0, j].set_title(f"y={y[j].item()} pred={p1[j].item()}", fontsize=9)
    axes[1, j].imshow(im1); axes[1, j].imshow(cam1[j].cpu().numpy(), cmap='jet', alpha=0.5)
    axes[2, j].imshow(im2); axes[2, j].set_title(f"pred={p2[j].item()}", fontsize=9)
    axes[3, j].imshow(im2); axes[3, j].imshow(cam2[j].cpu().numpy(), cmap='jet', alpha=0.5)
    for r in range(4): axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
for r, t in enumerate(['input', 'Grad-CAM (true class)', 'colors swapped', 'Grad-CAM (predicted)']):
    axes[r, 0].set_ylabel(t, fontsize=9)
plt.suptitle("Grad-CAM of the baseline model (conv3)"); plt.tight_layout()
plt.savefig(f"{OUT}/fig4_gradcam.png", dpi=150); plt.show()

## وظیفه ۴: پنج روش کاهش سوگیری

In [ ]:
# ---------------- وظیفه ۴: روش‌های کاهش سوگیری (هر پنج روش) ----------------
# روش ۱: Data Augmentation — ۱۰٪ نمونه بدون سوگیری (ρ=0.5) از ارقام «آموزش» ساخته و اضافه می‌شود
n_aug   = int(AUG_FRAC * len(S['train']))
aug_sel = np.random.RandomState(100).choice(len(tr_idx), n_aug, replace=False)
g_aug, y_aug = G_tr[tr_idx][aug_sel], Y_tr[tr_idx][aug_sel]
S_aug = CData(np.concatenate([G_tr[tr_idx], g_aug]),
              np.concatenate([Y_tr[tr_idx], y_aug]),
              np.concatenate([make_colors(Y_tr[tr_idx], 0.9, 42), make_colors(y_aug, 0.5, 47)]))
print("augmented train size:", len(S_aug), "| aligned fraction: %.3f" % S_aug.aligned.float().mean().item())

# روش ۲: Reweighting — w_i = 1 / P(color=c_i | shape=s_i)، با برآورد تجربی P از داده آموزش
def reweight_weights(tr):
    w = torch.zeros(len(tr), device=DEVICE)
    for hf in (0, 1):
        for c in (0, 1):
            m = (tr.half == hf) & (tr.c == c)
            p = m.sum().item() / (tr.half == hf).sum().item()
            w[m] = 1.0 / p
    return w / w.mean()                                   # نرمال‌سازی: میانگین وزن = ۱
W_RW = reweight_weights(S['train'])
print("reweighting: weight(aligned)=%.3f  weight(conflicting)=%.3f" %
      (W_RW[S['train'].aligned][0].item(), W_RW[~S['train'].aligned][0].item()))

# روش ۳: Group DRO — کمینه‌سازی بدترین خطای گروهی با وزن‌های q_g (به‌روزرسانی نمایی)
def fit_group_dro(tr, seed, eta=DRO_ETA, epochs=EPOCHS):
    set_seed(seed)
    model = SimpleCNN().to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=LR)
    q = torch.ones(4, device=DEVICE) / 4
    h, N = new_hist(), len(tr)
    for ep in range(epochs):
        model.train(); perm = torch.randperm(N, device=DEVICE)
        run, nb = torch.zeros((), device=DEVICE), 0
        for i in range(0, N, BATCH):
            idx = perm[i:i+BATCH]
            ce  = F.cross_entropy(model(tr.x[idx]), tr.y[idx], reduction='none')
            oh  = F.one_hot(tr.g[idx], 4).float()                    # (B,4)
            cnt = oh.sum(0)
            gl  = (oh * ce[:, None]).sum(0) / cnt.clamp_min(1)       # میانگین خطا در هر گروه (گروه غایب → 0)
            q   = q * torch.exp(eta * gl.detach())                   # گروه غایب: exp(0)=1
            q   = q / q.sum()
            loss = (q * gl).sum()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            run += loss.detach(); nb += 1
        log_epoch(h, model, (run / nb).item(), 'GroupDRO', ep)
    h['q_final'] = q.tolist()
    return model, h

# روش ۴: Learning from Failure — مدل سوگیرانه با GCE + مدل نهایی با وزن W = CE_B / (CE_B + CE_D)
def fit_lff(tr, seed, q=GCE_Q, alpha=EMA_ALPHA, epochs=EPOCHS):
    set_seed(seed)
    mb, md = SimpleCNN().to(DEVICE), SimpleCNN().to(DEVICE)
    opt = torch.optim.Adam(list(mb.parameters()) + list(md.parameters()), lr=LR)
    N = len(tr); h = new_hist()
    ema_b, ema_d = torch.zeros(N, device=DEVICE), torch.zeros(N, device=DEVICE)
    for ep in range(epochs):
        mb.train(); md.train(); perm = torch.randperm(N, device=DEVICE)
        run, nb = torch.zeros((), device=DEVICE), 0
        for i in range(0, N, BATCH):
            idx = perm[i:i+BATCH]; x, y = tr.x[idx], tr.y[idx]
            lb, ld = mb(x), md(x)
            ce_b = F.cross_entropy(lb, y, reduction='none'); ce_d = F.cross_entropy(ld, y, reduction='none')
            ema_b[idx] = alpha * ema_b[idx] + (1 - alpha) * ce_b.detach()       # EMA خطای هر نمونه
            ema_d[idx] = alpha * ema_d[idx] + (1 - alpha) * ce_d.detach()
            mx_b = torch.zeros(10, device=DEVICE).scatter_reduce(0, tr.y, ema_b, reduce='amax', include_self=True)
            mx_d = torch.zeros(10, device=DEVICE).scatter_reduce(0, tr.y, ema_d, reduce='amax', include_self=True)
            nb_  = ema_b[idx] / mx_b[y].clamp_min(1e-8)                          # نرمال‌سازی بر اساس بیشینه کلاس
            nd_  = ema_d[idx] / mx_d[y].clamp_min(1e-8)
            w    = nb_ / (nb_ + nd_ + 1e-8)                                      # نمونه‌های «سخت برای مدل سوگیرانه» وزن بیشتر
            p_y  = F.softmax(lb, 1).gather(1, y[:, None]).squeeze(1).clamp_min(1e-8)
            gce  = (1 - p_y ** q) / q                                            # Generalized CE
            loss = gce.mean() + (w.detach() * ce_d).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            run += loss.detach(); nb += 1
        log_epoch(h, md, (run / nb).item(), 'LfF', ep)                           # مدل نهایی = مدل بی‌سوگیری md
    return md, h

# روش ۵: IRM — دو محیط با ρ=0.9 و ρ=0.5 (نیمی از ارقام آموزش در هر محیط) و جریمه IRMv1
n_half = len(tr_idx) // 2
ENVS = [make_cmnist(G_tr[tr_idx[:n_half]], Y_tr[tr_idx[:n_half]], 0.9, 48),
        make_cmnist(G_tr[tr_idx[n_half:]], Y_tr[tr_idx[n_half:]], 0.5, 49)]

def fit_irm(envs, seed, lam=IRM_LAMBDA, anneal=IRM_ANNEAL, epochs=EPOCHS):
    set_seed(seed)
    model = SimpleCNN().to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=LR)
    h = new_hist(); n_steps = math.ceil(min(len(e) for e in envs) / BATCH)
    for ep in range(epochs):
        model.train(); perms = [torch.randperm(len(e), device=DEVICE) for e in envs]
        run = torch.zeros((), device=DEVICE)
        for s in range(n_steps):
            risks, pens = [], []
            for e, pm in zip(envs, perms):
                idx   = pm[s*BATCH:(s+1)*BATCH]
                scale = torch.tensor(1.0, device=DEVICE, requires_grad=True)       # w=1 در IRMv1
                risk  = F.cross_entropy(model(e.x[idx]) * scale, e.y[idx])
                gw,   = torch.autograd.grad(risk, [scale], create_graph=True)
                risks.append(risk); pens.append(gw ** 2)
            erm, pen = torch.stack(risks).mean(), torch.stack(pens).mean()
            wt   = lam if ep >= anneal else 1.0
            loss = erm + wt * pen
            if wt > 1.0: loss = loss / wt                                           # حفظ مقیاس گرادیان (مثل کد IRM)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            run += loss.detach()
        log_epoch(h, model, (run / n_steps).item(), 'IRM', ep)
    return model, h

METHODS = {
    'M1_Augmentation': lambda s: fit_erm(S_aug, s, tag='Aug'),
    'M2_Reweighting':  lambda s: fit_erm(S['train'], s, sample_w=W_RW, tag='Reweight'),
    'M3_GroupDRO':     lambda s: fit_group_dro(S['train'], s),
    'M4_LfF':          lambda s: fit_lff(S['train'], s),
    'M5_IRM':          lambda s: fit_irm(ENVS, s),
}

In [ ]:
# ---------------- اجرای همه روش‌ها روی همه seedها ----------------
for name, fn in METHODS.items():
    t0 = time.time()
    for s in SEEDS:
        m, h = fn(s)
        register(name, s, m, h)
        CUE.setdefault(name, {})[s] = cue_suite(m)
    print(f"##### {name} done in {time.time()-t0:.0f}s\n")
    torch.save({s: MODELS[name][s].state_dict() for s in SEEDS}, f"{OUT}/{name}_weights.pth")
    save_results()
CUE['Baseline'] = {s: cue_suite(MODELS['Baseline'][s]) for s in SEEDS}
save_results()

### جدول ۳ و ۴ و نمودارها

In [ ]:
# ---------------- جدول ۳: مقایسه روش‌ها ----------------
ORDER = ['Baseline'] + list(METHODS.keys())
def gap_stat(method, ood):
    v = np.array([RES[method][s]['id'] - RES[method][s][ood] for s in SEEDS]) * 100
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)

rows = []
for mth in ORDER:
    r = {'روش': mth}
    for k, nm in [('id', 'ID'), ('ood1', 'OOD-1'), ('ood2', 'OOD-2')]:
        m, sd = agg(mth, k); r[f'دقت {nm} %'] = f"{m:.2f} ± {sd:.2f}"
    for ood, nm in [('ood1', 'ID−OOD1'), ('ood2', 'ID−OOD2')]:
        m, sd = gap_stat(mth, ood); r[f'اختلاف {nm}'] = f"{m:.2f} ± {sd:.2f}"
    rows.append(r)
T3 = pd.DataFrame(rows); T3.to_csv(f"{OUT}/table3_methods.csv", index=False, encoding='utf-8-sig'); display(T3)

# جدول ۴ (تکمیلی): پروفایل «تکیه بر رنگ» برای همه روش‌ها
rows = []
for mth in ORDER:
    a, _ = cagg(mth, 'A_swap', 'acc'); b, _ = cagg(mth, 'B_gray_sum', 'acc'); f, _ = cagg(mth, 'M_shape_shuffle', 'follow')
    rows.append({'روش': mth, 'دقت پس از تعویض رنگ %': f"{a:.1f}", 'دقت Grayscale (حفظ شدت) %': f"{b:.1f}",
                 'نرخ پیروی از رنگ پس از بُر زدن شکل %': f"{f:.1f}"})
T4 = pd.DataFrame(rows); T4.to_csv(f"{OUT}/table4_cue_profile.csv", index=False, encoding='utf-8-sig'); display(T4)

# نمودار مقایسه‌ای
fig, ax = plt.subplots(figsize=(11, 4.5)); wd = 0.26; xs_ = np.arange(len(ORDER))
for j, (k, nm) in enumerate([('id', 'ID'), ('ood1', 'OOD-1'), ('ood2', 'OOD-2')]):
    ax.bar(xs_ + (j - 1) * wd, [agg(m_, k)[0] for m_ in ORDER], wd, yerr=[agg(m_, k)[1] for m_ in ORDER], capsize=3, label=nm)
ax.set_xticks(xs_); ax.set_xticklabels([m_.replace('_', '\n') for m_ in ORDER], fontsize=8)
ax.set_ylabel('accuracy (%)'); ax.set_ylim(0, 105); ax.legend(); ax.set_title('Comparison of debiasing methods')
plt.tight_layout(); plt.savefig(f"{OUT}/fig5_methods_comparison.png", dpi=150); plt.show()

# پویایی یادگیری (seed اول)
fig, ax = plt.subplots(1, 3, figsize=(15, 4)); s0 = SEEDS[0]
for j, (k, nm) in enumerate([('val_acc', 'validation (rho=0.9)'), ('ood1_acc', 'OOD-1 (rho=0.5)'), ('ood2_acc', 'OOD-2 (rho=0.1)')]):
    for mth in ORDER: ax[j].plot(range(1, EPOCHS + 1), HIST[mth][s0][k], label=mth)
    ax[j].set_title(nm); ax[j].set_xlabel('epoch'); ax[j].set_ylabel('accuracy')
ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT}/fig6_training_dynamics.png", dpi=150); plt.show()
save_results()

## گزارش Word از نتایج همین اجرا

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-docx'], check=True)

# -*- coding: utf-8 -*-
# Builds the Persian (RTL) Word report from results/results.json + figures. All numbers come from the file.
import json, os, re
import numpy as np
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_COLOR_INDEX
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

FA_FONT, LAT_FONT = "B Nazanin", "Times New Roman"
PN = {'Baseline': 'Baseline', 'M1_Augmentation': 'روش ۱ – Data Augmentation', 'M2_Reweighting': 'روش ۲ – Reweighting',
      'M3_GroupDRO': 'روش ۳ – Group DRO', 'M4_LfF': 'روش ۴ – LfF', 'M5_IRM': 'روش ۵ – IRM'}
FA_DIG = str.maketrans("0123456789", "۰۱۲۳۴۵۶۷۸۹")

_PPR_AFTER_BIDI = ['w:adjustRightInd', 'w:snapToGrid', 'w:spacing', 'w:ind', 'w:contextualSpacing', 'w:mirrorIndents',
    'w:suppressOverlap', 'w:jc', 'w:textDirection', 'w:textAlignment', 'w:textboxTightWrap', 'w:outlineLvl', 'w:divId',
    'w:cnfStyle', 'w:rPr', 'w:sectPr', 'w:pPrChange']
_RPR_AFTER_SZCS = ['w:highlight', 'w:u', 'w:effect', 'w:bdr', 'w:shd', 'w:fitText', 'w:vertAlign', 'w:rtl', 'w:cs', 'w:em',
    'w:lang', 'w:eastAsianLayout', 'w:specVanish', 'w:oMath']
_TBLPR_AFTER_BIDIV = ['w:tblStyleRowBandSize', 'w:tblStyleColBandSize', 'w:tblW', 'w:jc', 'w:tblCellSpacing', 'w:tblInd',
    'w:tblBorders', 'w:shd', 'w:tblLayout', 'w:tblCellMar', 'w:tblLook', 'w:tblCaption', 'w:tblDescription', 'w:tblPrChange']

def _style_fonts(style, size, bold=None):
    style.font.name = LAT_FONT; style.font.size = Pt(size)
    if bold is not None: style.font.bold = bold
    rpr = style.element.get_or_add_rPr(); rf = rpr.find(qn('w:rFonts'))
    for a in ('asciiTheme', 'hAnsiTheme', 'eastAsiaTheme', 'cstheme'): rf.attrib.pop(qn('w:' + a), None)
    rf.set(qn('w:cs'), FA_FONT)
    if rpr.find(qn('w:szCs')) is None:
        e = OxmlElement('w:szCs'); e.set(qn('w:val'), str(int(size * 2)))
        rpr.insert_element_before(e, *_RPR_AFTER_SZCS)
    if bold:
        if rpr.find(qn('w:bCs')) is None:
            e = OxmlElement('w:bCs'); rpr.insert_element_before(e, 'w:i', 'w:iCs', 'w:color', 'w:sz', 'w:szCs', *_RPR_AFTER_SZCS)
    if style.font.color is not None and style.name.startswith('Heading'):
        from docx.shared import RGBColor; style.font.color.rgb = RGBColor(0, 0, 0)

def _bidi(p):
    pPr = p._p.get_or_add_pPr()
    if pPr.find(qn('w:bidi')) is None:
        pPr.insert_element_before(OxmlElement('w:bidi'), *_PPR_AFTER_BIDI)

def _run(p, text, bold=False, italic=False, size=None, rtl=True, hl=False, latin=False):
    r = p.add_run(text)
    if rtl: r.font.rtl = True
    if bold: r.font.bold = True; r.font.cs_bold = True
    if italic: r.font.italic = True; r.font.cs_italic = True
    if size:
        r.font.size = Pt(size)
        e = OxmlElement('w:szCs'); e.set(qn('w:val'), str(int(size * 2)))
        r._r.get_or_add_rPr().insert_element_before(e, *_RPR_AFTER_SZCS)
    if hl: r.font.highlight_color = WD_COLOR_INDEX.YELLOW
    if latin: r.font.name = LAT_FONT
    return r

class Rep:
    def __init__(self):
        d = self.d = Document(); sec = d.sections[0]
        sec.page_width, sec.page_height = Cm(21), Cm(29.7)
        sec.left_margin = sec.right_margin = Cm(2.5); sec.top_margin = sec.bottom_margin = Cm(2.5)
        _style_fonts(d.styles['Normal'], 12)
        _style_fonts(d.styles['Heading 1'], 16, True); _style_fonts(d.styles['Heading 2'], 14, True)
        _style_fonts(d.styles['Heading 3'], 12, True)
        pf = d.styles['Normal'].paragraph_format; pf.space_after = Pt(6); pf.line_spacing = 1.25
        # شماره صفحه
        fp = sec.footer.paragraphs[0]; fp.alignment = WD_ALIGN_PARAGRAPH.CENTER
        f = OxmlElement('w:fldSimple'); f.set(qn('w:instr'), 'PAGE'); r = OxmlElement('w:r'); t = OxmlElement('w:t'); t.text = '1'
        r.append(t); f.append(r); fp._p.append(f)
    def h(self, text, level=1):
        p = self.d.add_heading(level=level); _bidi(p); _run(p, text, bold=True); return p
    def p(self, text='', bold=False, italic=False, center=False, size=None, hl=False):
        p = self.d.add_paragraph(); _bidi(p)
        if center: p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        else: p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
        _run(p, text, bold=bold, italic=italic, size=size, hl=hl); return p
    def bullets(self, items):
        for t in items:
            p = self.d.add_paragraph(); _bidi(p); p.paragraph_format.right_indent = Cm(0.6)
            _run(p, "• " + t)
    def formula(self, text):
        p = self.d.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        _run(p, text, italic=True, rtl=False, latin=True, size=11)
    def table(self, header, rows, caption=None, size=10):
        if caption: self.p(caption, bold=True, center=True, size=11)
        t = self.d.add_table(rows=1 + len(rows), cols=len(header)); t.style = 'Table Grid'
        t.alignment = WD_ALIGN_PARAGRAPH.CENTER
        tblPr = t._tbl.tblPr
        if tblPr.find(qn('w:bidiVisual')) is None: tblPr.insert_element_before(OxmlElement('w:bidiVisual'), *_TBLPR_AFTER_BIDIV)
        for i, row in enumerate([header] + rows):
            for j, val in enumerate(row):
                c = t.rows[i].cells[j]; para = c.paragraphs[0]; _bidi(para); para.alignment = WD_ALIGN_PARAGRAPH.CENTER
                sv = str(val)
                if re.fullmatch(r'[\d\.\s±+\-]+', sv.strip()) and i > 0:   # اعداد را LTR نگه دار تا «میانگین ± انحراف» جابه‌جا نشود
                    sv = '\u202a' + sv + '\u202c'
                _run(para, sv, bold=(i == 0), size=size)
                if i == 0:
                    shd = OxmlElement('w:shd'); shd.set(qn('w:val'), 'clear'); shd.set(qn('w:color'), 'auto'); shd.set(qn('w:fill'), 'D9D9D9')
                    c._tc.get_or_add_tcPr().append(shd)
        self.d.add_paragraph()
    def fig(self, path, caption, width=15.0):
        if not os.path.exists(path):
            self.p(f"[شکل یافت نشد: {os.path.basename(path)}]", hl=True); return
        self.d.add_picture(path, width=Cm(width)); self.d.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
        self.p(caption, italic=True, center=True, size=10)

def fa(n): return str(n).translate(FA_DIG)

def build_report(results_json, fig_dir, out_path):
    J = json.load(open(results_json, encoding='utf-8')); cfg, RES, CUE, HIST = J['cfg'], J['RES'], J['CUE'], J['HIST']
    SEEDS = [str(s) for s in cfg['SEEDS']]; ns = len(SEEDS); ORDER = [m for m in PN if m in RES]
    def arr(src, m, key): return np.array([src[m][s][key] for s in SEEDS]) * 100
    def ms(src, m, key):
        v = arr(src, m, key); return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)
    def pm(src, m, key, d=2): a, b = ms(src, m, key); return f"{a:.{d}f} ± {b:.{d}f}"
    def pc(m, probe, key, d=2):
        v = np.array([CUE[m][s][probe][key] for s in SEEDS]) * 100; return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)
    def pcs(m, probe, key, d=2): a, b = pc(m, probe, key); return f"{a:.{d}f} ± {b:.{d}f}"
    def gap(m, ood): v = np.array([RES[m][s]['id'] - RES[m][s][ood] for s in SEEDS]) * 100; return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)

    B = 'Baseline'; id_, o1, o2 = ms(RES, B, 'id')[0], ms(RES, B, 'ood1')[0], ms(RES, B, 'ood2')[0]
    d1, d2 = id_ - o1, id_ - o2
    lvl = 'شدید' if d2 >= 30 else ('متوسط' if d2 >= 10 else 'ضعیف')
    seed_txt = f"میانگین ± انحراف معیار روی {fa(ns)} seed مستقل" if ns > 1 else "نتیجه یک اجرا (یک seed)"
    R = Rep()

    # ---------- صفحه عنوان ----------
    R.p(); R.p("پروژه عملی هفته دوم", bold=True, center=True, size=20)
    R.p("تشخیص و کاهش همبستگی‌های کاذب در طبقه‌بندی تصاویر", bold=True, center=True, size=18)
    R.p("(مجموعه داده Colored MNIST)", center=True, size=14); R.p()
    R.p("نام اعضای تیم: ..............................", center=True); R.p("درس / استاد: ..............................", center=True)
    R.p("مهر ۱۴۰۵", center=True); R.d.add_page_break()

    # ---------- ۱. مقدمه ----------
    R.h("۱. مقدمه")
    R.p("مدل‌های یادگیری عمیق برای کمینه‌کردن خطای تجربی روی داده آموزش بهینه می‌شوند و هیچ تضمینی وجود ندارد که برای این کار از "
        "همان نشانه‌هایی استفاده کنند که طراح مدل در نظر داشته است. اگر در داده آموزش ویژگی‌ای غیرعلّی (مانند رنگ، بافت یا زمینه تصویر) با برچسب "
        "همبستگی داشته باشد، مدل می‌تواند به‌جای یادگیری نشانه وظیفه (Task Cue)، از این نشانه کاذب (Bias Cue) به‌عنوان میان‌بر استفاده کند. "
        "چنین مدلی روی داده هم‌توزیع (ID) عملکرد بسیار خوبی دارد، اما وقتی توزیع تغییر کند (OOD)، مثلاً همبستگی کاذب ضعیف یا معکوس شود، "
        "ممکن است دچار افت شدید شود. به همین دلیل پرسش ارزیابی از «مدل چقدر درست پیش‌بینی می‌کند؟» به «مدل چگونه و بر پایه کدام نشانه پیش‌بینی می‌کند؟» تغییر می‌کند.")
    R.p("اهمیت OOD در کاربردهای واقعی (پزشکی، خودرانی، تصمیم‌گیری‌های حساس) از آن‌جاست که توزیع داده در زمان استقرار تقریباً هرگز با توزیع آموزش یکسان نیست و "
        "دقت بالا روی داده هم‌توزیع، دلیلی بر اتکای مدل به نشانه‌های درست نیست.")
    R.p("اهداف این پروژه عبارت‌اند از:"); R.bullets([
        "ساخت مجموعه داده Colored MNIST با همبستگی کاذب کنترل‌شده (رنگ = نشانه کاذب، شکل رقم = نشانه وظیفه)؛",
        "آموزش یک CNN پایه و ارزیابی آن روی داده ID و دو مجموعه OOD؛",
        "تشخیص نشانه مورد اتکای مدل با آزمون‌های Counterfactual، Saliency Map، Grad-CAM و آزمون Misspecification؛",
        "پیاده‌سازی و مقایسه پنج روش کاهش همبستگی کاذب (Augmentation، Reweighting، Group DRO، LfF، IRM) و تحلیل علت موفقیت یا شکست آن‌ها."])

    # ---------- ۲. روش‌شناسی ----------
    R.h("۲. روش‌شناسی")
    R.h("۲.۱. ساخت داده", 2)
    R.p("هر تصویر MNIST به یک تصویر RGB با اندازه ۲۸×۲۸ تبدیل می‌شود که رقم فقط در یک کانال (قرمز یا سبز) قرار دارد. "
        "با احتمال ρ، ارقام ۰–۴ قرمز و ارقام ۵–۹ سبز می‌شوند و با احتمال ۱−ρ رنگ برعکس می‌شود. "
        "بنابراین شکل رقم نشانه وظیفه و رنگ نشانه کاذب است. تعداد و ρ مجموعه‌ها در جدول زیر آمده است.")
    R.table(["مجموعه", "ρ", "تعداد", "نقش"], [["آموزش", "0.9", "50,000", "آموزش مدل"], ["اعتبارسنجی", "0.9", "10,000", "پایش آموزش"],
            ["تست ID", "0.9", "10,000", "عملکرد هم‌توزیع"], ["تست OOD-1", "0.5", "10,000", "رنگ مستقل از شکل (بدون سوگیری)"],
            ["تست OOD-2", "0.1", "10,000", "همبستگی معکوس"]], caption="جدول ۰ – مشخصات مجموعه‌ها")
    R.p("ارقام آموزش و اعتبارسنجی از ۶۰٬۰۰۰ تصویر train در MNIST و به‌صورت کاملاً جدا انتخاب شده‌اند و ارقام تست از ۱۰٬۰۰۰ تصویر test می‌آیند؛ بنابراین نشت داده وجود ندارد. "
        "نکته: MNIST فقط ۱۰٬۰۰۰ تصویر test دارد، در حالی‌که صورت پروژه ۱۰٬۰۰۰ نمونه برای هر یک از سه مجموعه تست خواسته است (برش‌های [10000:20000] و [20000:30000] "
        "در کد نمونه خالی می‌شوند). برای رعایت اندازه‌ها، هر سه مجموعه ID، OOD-1 و OOD-2 از همان ۱۰٬۰۰۰ رقم تست ساخته شده‌اند و فقط رنگ‌آمیزی (ρ و seed) در آن‌ها فرق می‌کند. "
        "این انتخاب مزیتی هم دارد: تفاوت دقت بین این مجموعه‌ها دقیقاً ناشی از رنگ است، نه از تفاوت ارقام.")
    R.fig(os.path.join(fig_dir, "fig0_dataset_samples.png"), "شکل ۰ – نمونه‌هایی از هر مجموعه (یک تصویر برای هر رقم)", 15)
    R.h("۲.۲. مدل و آموزش", 2)
    R.p(f"مدل: {cfg.get('ARCH_FA', 'SimpleCNN با سه لایه کانولوشن')}. "
        f"آموزش با بهینه‌ساز Adam، نرخ یادگیری {cfg['LR']}، اندازه دسته {cfg['BATCH']}، {fa(cfg['EPOCHS'])} epoch و تابع هزینه CrossEntropy انجام شد. "
        f"همه نتایج برای {fa(ns)} seed گزارش شده‌اند ({seed_txt}). وزن‌های نهایی (epoch آخر) مبنای ارزیابی هستند؛ هایپرپارامترها از پیش تعیین شده‌اند و روی داده OOD تنظیم نشده‌اند.")
    R.h("۲.۳. روش‌های تشخیص نشانه", 2)
    R.bullets([
        "Counterfactual – تست A: تعویض رنگ‌ها (قرمز↔سبز) در تصاویر تست ID؛ تست B: حذف رنگ با Grayscale (میانگین سه کانال). "
        "چون میانگین‌گیری شدت پیکسل‌ها را به یک‌سوم می‌رساند، یک کنترل B′ هم با حفظ شدت (جمع کانال‌ها) اضافه شد تا اثر «حذف رنگ» از اثر «تغییر شدت» جدا شود.",
        "Saliency Map: Gradient×Input نسبت به لاجیت کلاس درست، به‌همراه گرادیان خام به تفکیک کانال‌های R و G. چون در این داده پیکسل‌های رنگی همان پیکسل‌های رقم هستند، "
        "Gradient×Input به‌تنهایی نمی‌تواند رنگ را از شکل جدا کند؛ گرادیان کانال غیرفعال نشان می‌دهد مدل به رنگ مخالف چه‌قدر حساس است.",
        "Grad-CAM: روی خروجی آخرین لایه کانولوشن (نقشه ۷×۷ که به ۲۸×۲۸ بزرگ می‌شود)، یک‌بار برای تصویر اصلی و یک‌بار برای همان رقم با رنگ تعویض‌شده.",
        "تست Misspecification روی ۱۰۰۰ نمونه تست: تغییر نشانه شکل (چرخش ۹۰°، حذف نیمه بالایی، بُر زدن قطعه‌های ۷×۷) و تغییر نشانه رنگ (تعویض و حذف). "
        "از آن‌جا که تخریب شکل دقت ۱۰ کلاسه را برای هر مدلی کاهش می‌دهد، دو معیار تکمیلی تعریف شد: «دقت نیمه‌کلاس» (تشخیص اینکه رقم در ۰–۴ است یا ۵–۹) و «نرخ پیروی از رنگ» "
        "(درصد پیش‌بینی‌هایی که نیمه‌کلاسشان با نیمه‌ای که رنگ نمایش‌داده‌شده القا می‌کند یکی است). بالا ماندن نرخ پیروی از رنگ پس از تخریب شکل نشانه تکیه بر رنگ است."])
    R.h("۲.۴. روش‌های کاهش سوگیری", 2)
    R.p(f"روش ۱ – Data Augmentation: {fa(int(cfg['AUG_FRAC']*100))}٪ نمونه بدون سوگیری (ρ=0.5) از ارقام آموزش ساخته و به مجموعه آموزش افزوده می‌شود (اندازه ≈ {fa(int(cfg['N_TRAIN']*(1+cfg['AUG_FRAC'])))} نمونه). این نمونه‌ها ارقام جدید نیستند؛ فقط رنگ‌آمیزی متفاوتی از ارقام موجودند.")
    R.p("روش ۲ – Reweighting: به هر نمونه وزن زیر داده می‌شود و وزن‌ها طوری نرمال می‌شوند که میانگینشان ۱ باشد:")
    R.formula("w_i = 1 / P(color = c_i | shape = s_i)")
    R.p("احتمال‌ها به‌صورت تجربی از داده آموزش برآورد شده‌اند؛ نمونه‌های مخالف‌جهت (حدود ۱۰٪) وزنی تقریباً ۹ برابر نمونه‌های هم‌جهت می‌گیرند.")
    R.p("روش ۳ – Group DRO: چهار گروه (رنگ × نیمه‌کلاس) تعریف می‌شود و به‌جای میانگین خطا، خطای بدترین گروه کمینه می‌شود:")
    R.formula("R(θ) = max_g E_{(x,y)~P_g}[ ℓ(θ; x, y) ],   q_g ← q_g · exp(η_q · L_g),   q ← q / Σq")
    R.p(f"در عمل، در هر دسته میانگین خطای هر گروه محاسبه، وزن q_g با η_q = {cfg['DRO_ETA']} به‌روزرسانی و Σ q_g L_g کمینه می‌شود.")
    R.p("روش ۴ – Learning from Failure (LfF): یک مدل سوگیرانه با Generalized Cross Entropy آموزش می‌بیند که ویژگی‌های «آسان» را سریع‌تر می‌گیرد، و مدل نهایی با وزن‌دهی نمونه‌ها بر اساس خطای نسبی دو مدل آموزش می‌بیند:")
    R.formula("GCE(p, y) = (1 − p_y^q) / q,     W(x) = CE_B(x) / ( CE_B(x) + CE_D(x) ),     L = GCE_B + W(x) · CE_D")
    R.p(f"با q = {cfg['GCE_Q']} و میانگین متحرک نمایی خطای هر نمونه (α = {cfg['EMA_ALPHA']}) که بر بیشینه خطای کلاس نرمال می‌شود. این روش به برچسب رنگ نیاز ندارد.")
    R.p("روش ۵ – IRM: دو محیط تعریف شد (نیمی از ارقام آموزش با ρ=0.9 و نیم دیگر با ρ=0.5) و جریمه IRMv1 به هزینه افزوده شد:")
    R.formula("min_Φ  Σ_e R^e(Φ) + λ · ‖ ∇_{w|w=1} R^e(w · Φ) ‖²")
    R.p(f"در {fa(cfg['IRM_ANNEAL'])} epoch اول λ=1 و سپس λ={cfg['IRM_LAMBDA']:.0e} است و هزینه برای حفظ مقیاس گرادیان بر λ تقسیم می‌شود. توجه کنید که مجموعه آموزش IRM همبستگی کلی ≈ ۰.۷ دارد و به برچسب محیط نیاز دارد؛ بنابراین مستقیماً با Baseline هم‌سطح نیست.")
    R.h("۲.۵. معیارها", 2)
    R.p("معیار اصلی دقت ۱۰ کلاسه است. برای مقایسه روش‌ها اختلاف دقت ID و OOD (ID − OOD) نیز گزارش می‌شود. همه مقادیر درصد هستند و "
        f"به‌صورت {seed_txt} آمده‌اند.")

    # ---------- ۳. نتایج ----------
    R.h("۳. نتایج")
    R.h("۳.۱. مدل پایه و ارزیابی ID/OOD", 2)
    keys = [('train', 'آموزش', '0.9'), ('val', 'اعتبارسنجی', '0.9'), ('id', 'تست ID', '0.9'), ('ood1', 'تست OOD-1 (بدون سوگیری)', '0.5'), ('ood2', 'تست OOD-2 (معکوس)', '0.1')]
    R.table(["مجموعه", "ρ", "دقت %"], [[n, r, pm(RES, B, k)] for k, n, r in keys] +
            [["تست ID – رنگ هم‌جهت با شکل", "-", pm(RES, B, 'id_aligned')], ["تست ID – رنگ مخالف شکل", "-", pm(RES, B, 'id_conflict')]],
            caption="جدول ۱ – ارزیابی مدل پایه")
    R.p(f"دقت مدل پایه روی ID برابر {id_:.1f}٪، روی OOD-1 برابر {o1:.1f}٪ و روی OOD-2 برابر {o2:.1f}٪ است؛ یعنی افت {d1:.1f} واحد درصد در OOD-1 و {d2:.1f} واحد درصد در OOD-2 نسبت به ID. "
        f"بر اساس معیار این گزارش (افت OOD-2 بیش از ۳۰ واحد = شدید، ۱۰ تا ۳۰ = متوسط، کمتر از ۱۰ = ضعیف) شدت وابستگی مدل پایه به رنگ «{lvl}» ارزیابی می‌شود.")
    R.fig(os.path.join(fig_dir, "fig1_baseline_training_curves.png"), "شکل ۱ – منحنی Loss و Accuracy مدل پایه (آموزش و اعتبارسنجی)", 15)
    R.fig(os.path.join(fig_dir, "fig2_baseline_accuracy.png"), "شکل ۲ – دقت مدل پایه روی پنج مجموعه", 11)

    R.h("۳.۲. تشخیص نشانه", 2)
    def interp(drop): return "کاهش معنادار ⇒ تکیه بر رنگ" if drop >= 10 else ("کاهش اندک" if drop >= 3 else "بدون کاهش مهم ⇒ عدم تکیه بر رنگ")
    ref = pc(B, 'original', 'acc')[0]; rows2 = []
    for k, nm in [('original', 'تست ID اصلی (مرجع)'), ('A_swap', 'تست A (تعویض رنگ)'), ('B_gray_mean', 'تست B (Grayscale – میانگین)'), ('B_gray_sum', 'تست B′ (Grayscale – حفظ شدت)')]:
        a = pc(B, k, 'acc')[0]; rows2.append([nm, pcs(B, k, 'acc'), f"{pc(B, k, 'follow')[0]:.1f}", "-" if k == 'original' else interp(ref - a)])
    R.table(["تست", "دقت %", "نرخ پیروی از رنگ %", "تفسیر (بر پایه افت دقت)"], rows2, caption="جدول ۲ – آزمون‌های Counterfactual برای مدل پایه")
    A = pc(B, 'A_swap', 'acc')[0]; Bm = pc(B, 'B_gray_mean', 'acc')[0]; Bs = pc(B, 'B_gray_sum', 'acc')[0]
    R.p(f"با تعویض رنگ، دقت از {ref:.1f}٪ به {A:.1f}٪ می‌رسد (تغییر: {A-ref:+.1f} واحد درصد). با Grayscale میانگین‌گیری دقت {Bm:.1f}٪ و با Grayscale حفظ‌کننده شدت {Bs:.1f}٪ است. "
        + ("اختلاف قابل‌توجه بین B و B′ نشان می‌دهد بخشی از افت در تست B به کاهش شدت تصویر (و نه صرفاً حذف رنگ) مربوط است. " if abs(Bm - Bs) >= 5 else "نزدیک بودن B و B′ نشان می‌دهد تغییر شدت اثر مهمی نداشته است. "))
    rows2b = []
    for k, nm in [('M_orig', 'بدون تغییر'), ('M_shape_rot90', 'تغییر شکل: چرخش ۹۰°'), ('M_shape_occlude', 'تغییر شکل: حذف نیمه بالایی'), ('M_shape_shuffle', 'تغییر شکل: بُر زدن قطعه‌ها'),
                  ('M_color_swap', 'تغییر رنگ: تعویض'), ('M_color_gray', 'تغییر رنگ: حذف (حفظ شدت)')]:
        rows2b.append([nm, pcs(B, k, 'acc'), f"{pc(B, k, 'half_acc')[0]:.1f}", f"{pc(B, k, 'follow')[0]:.1f}"])
    R.table(["آزمون (۱۰۰۰ نمونه)", "دقت ۱۰ کلاسه %", "دقت نیمه‌کلاس %", "نرخ پیروی از رنگ %"], rows2b, caption="جدول ۲ب – آزمون Misspecification برای مدل پایه")
    sh_f, sh_h = pc(B, 'M_shape_shuffle', 'follow')[0], pc(B, 'M_shape_shuffle', 'half_acc')[0]
    R.p(f"پس از بُر زدن قطعه‌های تصویر (شکل تخریب شده ولی رنگ حفظ شده) نرخ پیروی از رنگ {sh_f:.1f}٪ است؛ مقدار حدود ۵۰٪ یعنی پیش‌بینی نیمه‌کلاس مستقل از رنگ است و مقدار نزدیک ۱۰۰٪ یعنی مدل تقریباً فقط رنگ را دنبال می‌کند.")
    R.fig(os.path.join(fig_dir, "fig3_saliency.png"), "شکل ۳ – Saliency Map (Gradient×Input و گرادیان کانال‌های R و G) برای یک نمونه از هر کلاس؛ ۱۰ نمونه از هر کلاس در فایل fig3b_saliency_100samples.png", 16.5)
    R.fig(os.path.join(fig_dir, "fig4_gradcam.png"), "شکل ۴ – Grad-CAM برای تصویر اصلی و همان رقم با رنگ تعویض‌شده", 16.5)
    R.p("تحلیل کیفی شکل‌های ۳ و ۴ (پس از مشاهده شکل‌ها توسط تیم تکمیل شود): نقشه‌ها روی شکل رقم متمرکزند یا روی رنگ؟ پس از تعویض رنگ، پیش‌بینی و ناحیه مورد توجه Grad-CAM چه تغییری می‌کند؟ "
        "(یادآوری: چون در این داده پیکسل رقم همان پیکسل رنگی است، تمرکز Saliency روی پیکسل‌های رقم به‌تنهایی دلیل تکیه بر شکل نیست؛ باید ردیف گرادیان کانال R/G و نتیجه تعویض رنگ را هم در نظر گرفت.)", hl=True)

    R.h("۳.۳. مقایسه روش‌های کاهش سوگیری", 2)
    rows3 = []
    for m in ORDER:
        rows3.append([PN[m], pm(RES, m, 'id'), pm(RES, m, 'ood1'), pm(RES, m, 'ood2'), f"{gap(m,'ood1')[0]:.2f} ± {gap(m,'ood1')[1]:.2f}", f"{gap(m,'ood2')[0]:.2f} ± {gap(m,'ood2')[1]:.2f}"])
    R.table(["روش", "دقت ID", "دقت OOD-1", "دقت OOD-2", "اختلاف ID−OOD1", "اختلاف ID−OOD2"], rows3, caption="جدول ۳ – مقایسه روش‌ها (دقت به درصد)", size=9)
    R.fig(os.path.join(fig_dir, "fig5_methods_comparison.png"), "شکل ۵ – مقایسه دقت روش‌ها روی ID، OOD-1 و OOD-2", 16)
    rows4 = [[PN[m], f"{pc(m,'A_swap','acc')[0]:.1f}", f"{pc(m,'B_gray_sum','acc')[0]:.1f}", f"{pc(m,'M_shape_shuffle','follow')[0]:.1f}"] for m in ORDER]
    R.table(["روش", "دقت پس از تعویض رنگ %", "دقت Grayscale (حفظ شدت) %", "نرخ پیروی از رنگ پس از بُر زدن شکل %"], rows4, caption="جدول ۴ – پروفایل تکیه بر رنگ برای همه روش‌ها", size=9)
    R.fig(os.path.join(fig_dir, "fig6_training_dynamics.png"), "شکل ۶ – پویایی یادگیری: دقت اعتبارسنجی، OOD-1 و OOD-2 در طول epochها (فقط برای تحلیل؛ برای انتخاب مدل استفاده نشده است)", 16.5)

    # ---------- ۴. تحلیل ----------
    R.h("۴. تحلیل")
    R.h("۴.۱. چرا مدل پایه به رنگ تکیه می‌کند؟ (سوگیری سادگی)", 2)
    h0 = HIST[B][SEEDS[0]]; e1, eN = h0['ood2_acc'][0] * 100, h0['ood2_acc'][-1] * 100
    R.p("شبکه‌های عمیق با گرادیان کاهشی به‌طور ضمنی به‌سمت ویژگی‌های ساده‌تر سوگیری دارند (Simplicity Bias): رنگ یک ویژگی خطی‌جداپذیر و یک‌کاناله است که با ۹۰٪ احتمال نیمه‌کلاس را مشخص می‌کند، "
        "در حالی‌که شکل رقم ویژگی پیچیده‌ای است که باید از ترکیب لبه‌ها و منحنی‌ها ساخته شود. بنابراین رنگ سریع‌تر یاد گرفته می‌شود و تا زمانی که هزینه کاهش می‌یابد، مدل انگیزه‌ای برای کنار گذاشتن آن ندارد.")
    if lvl in ('شدید', 'متوسط'):
        R.p(f"نتایج این پروژه با این پیش‌بینی سازگار است: افت {d2:.1f} واحدی در OOD-2 و افت دقت پس از تعویض رنگ ({ref:.1f}٪ ← {A:.1f}٪) نشان می‌دهد مدل پایه، دست‌کم تا حدی، تصمیم خود را بر رنگ بنا کرده است.")
    else:
        R.p(f"برخلاف پیش‌بینی صورت پروژه، در این پیکربندی مدل پایه تنها وابستگی «{lvl}» به رنگ نشان داد (افت OOD-2 برابر {d2:.1f} واحد). یک توضیح محتمل این است که رنگ فقط اطلاعات "
            "دودویی (نیمه‌کلاس) می‌دهد، در حالی‌که برای تفکیک ۱۰ کلاس شکل رقم لازم است و حدود ۱۰٪ نمونه‌های مخالف‌جهت (≈ ۵۰۰۰ تصویر) سیگنال مستقیمی علیه میان‌بر رنگ فراهم می‌کنند. این توضیح فرضیه است و در این پروژه مستقیماً آزمون نشده است.")
    if e1 < eN - 5:
        R.p(f"پویایی آموزش نیز با سوگیری سادگی هم‌خوان است: در seed اول، دقت OOD-2 پس از epoch اول {e1:.1f}٪ بود و تا پایان آموزش به {eN:.1f}٪ رسید (شکل ۶)؛ یعنی مدل ابتدا به نشانه‌ای تکیه می‌کند که روی OOD-2 بدتر عمل می‌کند و بعداً تا حدی اصلاح می‌شود.")
    else:
        R.p(f"در منحنی‌های آموزش، دقت OOD-2 در seed اول از {e1:.1f}٪ (epoch اول) به {eN:.1f}٪ (epoch آخر) رسید؛ بنابراین شواهد پویایی برای «تکیه اولیه و سپس اصلاح» در این اجرا قوی نیست.")
    R.h("۴.۲. چرا OOD-2 سخت‌ترین مجموعه است؟ (همبستگی معکوس)", 2)
    R.p(f"در OOD-2 فقط ۱۰٪ نمونه‌ها رنگ هم‌جهت با شکل دارند، یعنی رنگ نه‌تنها بی‌فایده بلکه گمراه‌کننده است: هر چه مدل بیشتر به رنگ تکیه کرده باشد، پیش‌بینی‌هایش را در جهت اشتباه می‌برد. "
        f"در OOD-1 رنگ مستقل از شکل است و تکیه بر آن فقط اطلاعات را از دست می‌دهد (به‌طور میانگین ۵۰٪ نمونه‌ها هم‌جهت‌اند). در نتیجه انتظار نظری این است که دقت OOD-2 از OOD-1 و دقت OOD-1 از ID کمتر (یا حداکثر برابر) باشد. "
        f"در این اجرا: ID = {id_:.1f}٪، OOD-1 = {o1:.1f}٪ و OOD-2 = {o2:.1f}٪"
        + (" که با این ترتیب سازگار است." if (o2 <= o1 + 0.5 and o1 <= id_ + 0.5) else " که با این ترتیب ساده سازگار نیست و باید علتش بررسی شود."))
    R.p(f"مقایسه دقت روی نمونه‌های هم‌جهت و مخالف‌جهت در ID نیز همین را نشان می‌دهد: دقت روی نمونه‌های هم‌جهت {ms(RES,B,'id_aligned')[0]:.1f}٪ و روی نمونه‌های مخالف‌جهت {ms(RES,B,'id_conflict')[0]:.1f}٪ است. "
        "اختلاف این دو عدد معیار مستقیمی از «بهره‌ای» است که مدل از همبستگی کاذب می‌برد.")
    R.h("۴.۳. چرا برخی روش‌ها بهتر کار می‌کنند؟ (تحلیل نظری و نتایج)", 2)
    th = {'M1_Augmentation': "با افزودن نمونه‌های مستقل از رنگ، همبستگی کلی رنگ–برچسب در داده آموزش کاهش می‌یابد و شکل نسبت به رنگ نشانه‌ای پایدارتر برای کمینه‌کردن هزینه می‌شود؛ این روش ساده‌ترین و معمولاً مؤثرترین راه است، اما به برچسب رنگ برای ساخت نمونه‌ها و توان بازرنگ‌آمیزی داده نیاز دارد.",
          'M2_Reweighting': "وزن‌دهی عکس احتمال شرطی رنگ، توزیع آموزش را در عمل به توزیعی تبدیل می‌کند که رنگ و شکل در آن مستقل‌اند (مقدار مورد انتظار هزینه تحت P(c|s) مستقل می‌شود)؛ هزینه آن واریانس بالاتر گرادیان است، زیرا نمونه‌های کم‌شمار وزن زیادی دارند.",
          'M3_GroupDRO': "کمینه‌کردن بدترین خطای گروهی، گروه‌های اقلیت (رنگ مخالف شکل) را با وزن بیشتر وارد آموزش می‌کند. نظریه و نتایج Sagawa و همکاران نشان می‌دهند که برای مدل‌های بیش‌پارامتری که داده آموزش را کاملاً حفظ می‌کنند، این روش بدون منظم‌سازی قوی (L2 یا توقف زودهنگام) اثر محدودی دارد؛ در این پروژه چنین منظم‌سازی‌ای تنظیم نشد.",
          'M4_LfF': "مدل سوگیرانه (GCE) تمایل دارد نمونه‌های آسان و هم‌جهت را سریع یاد بگیرد؛ نمونه‌هایی که این مدل روی آن‌ها خطای زیاد دارد (احتمالاً مخالف‌جهت) وزن بیشتری برای مدل نهایی می‌گیرند. مزیت اصلی این است که به برچسب رنگ نیاز ندارد؛ ضعف آن حساسیت به هایپرپارامترها و اینکه نمونه‌های پرنویز نیز ممکن است وزن زیادی بگیرند.",
          'M5_IRM': "IRM به‌دنبال نمایشی است که پیش‌بین بهینه روی آن در همه محیط‌ها یکسان باشد؛ چون رابطه رنگ–برچسب بین محیط‌ها (ρ=0.9 و ρ=0.5) متفاوت است، رنگ نمی‌تواند چنین نمایشی بسازد. در عمل بهینه‌سازی IRMv1 غیرمحدب و به مقدار λ، زمان‌بندی جریمه و تعداد محیط‌ها بسیار حساس است."}
    base_m = {k: ms(RES, B, k) for k in ('id', 'ood1', 'ood2')}
    for m in ORDER[1:]:
        o1m, o2m, idm = ms(RES, m, 'ood1'), ms(RES, m, 'ood2'), ms(RES, m, 'id')
        dd1, dd2 = o1m[0] - base_m['ood1'][0], o2m[0] - base_m['ood2'][0]
        noise = max(base_m['ood2'][1], o2m[1]); noise1 = max(base_m['ood1'][1], o1m[1])
        verdict = ("بهبود" if dd2 > max(noise, 1.0) else ("افت" if dd2 < -max(noise, 1.0) else "تفاوتی فراتر از نوسان بین seedها"))
        R.p(f"{PN[m]}: دقت ID از {base_m['id'][0]:.1f}٪ به {idm[0]:.1f}٪، OOD-1 از {base_m['ood1'][0]:.1f}٪ به {o1m[0]:.1f}٪ ({dd1:+.1f}) و OOD-2 از {base_m['ood2'][0]:.1f}٪ به {o2m[0]:.1f}٪ ({dd2:+.1f}) تغییر کرد؛ "
            f"در OOD-2 نتیجه: {verdict}. {th[m]}")
    worst = {m: min(ms(RES, m, 'ood1')[0], ms(RES, m, 'ood2')[0]) for m in ORDER}
    bm = max(worst, key=worst.get)
    R.p(f"بر اساس بدترین دقت OOD (min(OOD-1, OOD-2))، بهترین روش در این آزمایش «{PN[bm]}» با {worst[bm]:.1f}٪ و مدل پایه با {worst[B]:.1f}٪ است. "
        f"با توجه به اینکه تنها {fa(ns)} seed اجرا شده است، تفاوت‌های کوچک‌تر از انحراف معیار جدول ۳ نباید به‌عنوان برتری قطعی تفسیر شوند.")
    R.h("۴.۴. محدودیت‌های پروژه", 2)
    R.bullets([
        "MNIST رنگی یک مسئله اسباب‌بازی است؛ نتایج الزاماً به تصاویر طبیعی (مثلاً Waterbirds) تعمیم نمی‌یابد.",
        "نشانه کاذب تک‌بعدی و کاملاً مشخص است؛ در عمل ممکن است چندین نشانه کاذب ناشناخته وجود داشته باشد.",
        "سه مجموعه تست از ارقام یکسان ساخته شده‌اند (به دلیل محدودیت اندازه test در MNIST)؛ این کار مقایسه را دقیق‌تر می‌کند اما سه مجموعه از نظر ارقام مستقل نیستند.",
        "هایپرپارامترهای روش‌ها تنظیم نشده‌اند و مدل نهایی، وزن epoch آخر است؛ چون مجموعه اعتبارسنجی هم سوگیر (ρ=0.9) است، انتخاب مدل بر اساس آن می‌توانست مدل‌های وابسته‌تر به رنگ را ترجیح دهد.",
        "روش‌های ۲ و ۳ و ساخت محیط‌های IRM به برچسب نشانه کاذب (رنگ) نیاز دارند؛ این اطلاعات در بسیاری از کاربردها در دسترس نیست.",
        "محیط‌های IRM از بازرنگ‌آمیزی ارقام آموزش ساخته شدند و همبستگی کلی داده آن (≈ ۰.۷) با Baseline (۰.۹) یکسان نیست، پس مقایسه مستقیم با Baseline کاملاً هم‌سطح نیست.",
        f"تعداد seedها {fa(ns)} است و برآورد انحراف معیار با این تعداد نااطمینان است.",
        "Saliency و Grad-CAM ابزارهای توصیفی‌اند و در این داده به‌تنهایی رنگ را از شکل تفکیک نمی‌کنند؛ شواهد اصلی، آزمون‌های Counterfactual و Misspecification هستند."])

    # ---------- ۵. نتیجه‌گیری ----------
    R.h("۵. نتیجه‌گیری و پیشنهادها")
    R.p(f"مدل پایه روی ID به دقت {id_:.1f}٪ رسید ولی روی OOD-1 و OOD-2 به‌ترتیب {o1:.1f}٪ و {o2:.1f}٪ بود و وابستگی آن به رنگ در این پروژه «{lvl}» ارزیابی شد. "
        f"بهترین روش از نظر بدترین دقت OOD، «{PN[bm]}» بود. مهم‌ترین نکته روش‌شناختی این است که فقط گزارش دقت روی داده هم‌توزیع می‌تواند وابستگی به نشانه کاذب را پنهان کند و ارزیابی باید با مجموعه‌های OOD و آزمون‌های Counterfactual تکمیل شود.")
    R.p("پیشنهادها: (۱) افزایش تعداد seedها و گزارش بازه اطمینان؛ (۲) انتخاب مدل و تنظیم هایپرپارامتر با یک مجموعه اعتبارسنجی بدون سوگیری یا معیار دقت بدترین گروه؛ "
        "(۳) آزمون Group DRO با منظم‌سازی قوی‌تر؛ (۴) ارزیابی روی مجموعه‌های واقعی‌تر مانند Waterbirds یا CelebA؛ (۵) ترکیب روش‌ها (مثلاً Augmentation همراه با Reweighting).")

    # ---------- مراجع ----------
    R.h("مراجع")
    refs = ["[R1] Mucsányi et al. (2023), فصل ۲، صص ۱۴–۵۶ (مطابق ارجاع صورت پروژه).",
        "[R2] Varshney, K. R. (2022), Trustworthy Machine Learning, فصل ۹ (مطابق ارجاع صورت پروژه).",
        "[R3] Pruksachatkun et al. (2023), فصل ۴، صص ۱۸۱–۲۰۰ (مطابق ارجاع صورت پروژه).",
        "[4] Arjovsky, M., Bottou, L., Gulrajani, I., Lopez-Paz, D. (2019). Invariant Risk Minimization. arXiv:1907.02893.",
        "[5] Sagawa, S., Koh, P. W., Hashimoto, T., Liang, P. (2020). Distributionally Robust Neural Networks for Group Shifts. ICLR.",
        "[6] Nam, J., Cha, H., Ahn, S., Lee, J., Shin, J. (2020). Learning from Failure: Training Debiased Classifier from Biased Classifier. NeurIPS.",
        "[7] Bahng, H., Chun, S., Yun, S., Choo, J., Oh, S. J. (2020). Learning De-biased Representations with Biased Representations. ICML.",
        "[8] Geirhos, R. et al. (2019). ImageNet-trained CNNs are biased towards texture; increasing shape bias improves accuracy and robustness. ICLR.",
        "[9] Selvaraju, R. R. et al. (2017). Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization. ICCV.",
        "[10] Simonyan, K., Vedaldi, A., Zisserman, A. (2014). Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps. ICLR Workshop.",
        "[11] Zhang, Z., Sabuncu, M. (2018). Generalized Cross Entropy Loss for Training Deep Neural Networks with Noisy Labels. NeurIPS.",
        "[12] Shah, H. et al. (2020). The Pitfalls of Simplicity Bias in Neural Networks. NeurIPS."]
    for t in refs:
        p = R.d.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.LEFT if t[0:2] != '[R' else WD_ALIGN_PARAGRAPH.LEFT
        _run(p, t, rtl=False, latin=True, size=10)
    R.d.save(out_path); return out_path


build_report(f'{OUT}/results.json', OUT, f'{OUT}/report_light.docx')
print('report saved')

In [ ]:
import shutil
shutil.make_archive('project_light_results', 'zip', OUT)
try:
    from google.colab import files
    files.download('project_light_results.zip')
except Exception:
    print('zip saved: project_light_results.zip')